# Data Pipeline Sanity Check

Load one batch from the configured training and validation splits and visually inspect the food labels and Nutrition5k-derived targets before training. Nutrition targets are class-level modeling proxies, not Food-101 ground truth. This notebook does not train a model or download datasets.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
import yaml


cwd = Path.cwd().resolve()
ml_dir = None
for candidate in (cwd, *cwd.parents):
    if (candidate / "configs" / "base.yaml").is_file():
        ml_dir = candidate
        break
    if (candidate / "ml" / "configs" / "base.yaml").is_file():
        ml_dir = candidate / "ml"
        break
if ml_dir is None:
    raise FileNotFoundError("Could not locate ml/configs/base.yaml from the current directory or its parents.")
sys.path.insert(0, str(ml_dir.parent))

with (ml_dir / "configs" / "base.yaml").open(encoding="utf-8") as config_file:
    config = yaml.safe_load(config_file)

from ml.data.food_dataset import Food101NutritionDataset
from ml.data.splits import load_or_create_splits
from ml.preprocessing.transforms import get_eval_transform, get_train_transform


def ml_path(value):
    path = Path(value)
    return path if path.is_absolute() else ml_dir / path


dataset_config = config["dataset"]
split_config = config.get("data_splits", {})
splits_path = ml_path(split_config.get("manifest", "data/splits.json"))
print(f"ML directory: {ml_dir}")
print(f"Split manifest: {splits_path}")

## Load Resolved Samples and Reuse the Split Manifest

This builds the sample index without opening every image, then loads or creates the deterministic manifest from `data_splits` in `base.yaml`. Unmatched classes are skipped and summarized by the dataset class.

In [ ]:
identity_transform = lambda image: image
train_source = Food101NutritionDataset(split="train", transform=identity_transform)
test_source = Food101NutritionDataset(split="test", transform=identity_transform)
all_samples = train_source.samples + test_source.samples

split_manifest = load_or_create_splits(
    all_samples,
    splits_path,
    train_ratio=split_config.get("train_ratio", 0.70),
    val_ratio=split_config.get("val_ratio", 0.15),
    test_ratio=split_config.get("test_ratio", 0.15),
    seed=split_config.get("seed", 42),
)
print(f"Resolved Food-101 samples: {len(all_samples):,}")
for split_name, indices in split_manifest["split_indices"].items():
    print(f"{split_name}: {len(indices):,}")

## Build Train and Validation DataLoaders

A transform view applies augmentation only to training samples and deterministic preprocessing to validation samples while preserving the same saved global sample indices.

In [ ]:
class ManifestDataset(Dataset):
    def __init__(self, samples, indices, transform):
        self.samples = samples
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        image_path, class_index, nutrition_values = self.samples[self.indices[index]]
        if not image_path.is_file():
            raise FileNotFoundError(f"Image from the split manifest is missing: {image_path}")
        with Image.open(image_path) as image_file:
            image = image_file.convert("RGB")
        image_tensor = self.transform(image)
        nutrition_target = torch.tensor(nutrition_values, dtype=torch.float32)
        return image_tensor, class_index, nutrition_target


train_transform = get_train_transform(config.get("image_size", 224))
eval_transform = get_eval_transform(config.get("image_size", 224))
train_dataset = ManifestDataset(
    all_samples,
    split_manifest["split_indices"]["train"],
    train_transform,
)
val_dataset = ManifestDataset(
    all_samples,
    split_manifest["split_indices"]["val"],
    eval_transform,
)

batch_size = int(config.get("batch_size", 32))
seed = int(split_config.get("seed", 42))
train_generator = torch.Generator().manual_seed(seed)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    generator=train_generator,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)
print(f"Train DataLoader batches: {len(train_loader):,}")
print(f"Validation DataLoader batches: {len(val_loader):,}")

## Inspect One Batch From Each Loader

Images are unnormalized for display. Review whether the class and nutrition proxy shown above each image are plausible; investigate questionable labels or targets before training.

In [ ]:
train_batch = next(iter(train_loader))
val_batch = next(iter(val_loader))
for split_name, batch in (("train", train_batch), ("validation", val_batch)):
    images, class_indices, nutrition_targets = batch
    print(
        f"{split_name}: images={tuple(images.shape)}, "
        f"class_labels={tuple(class_indices.shape)}, "
        f"nutrition_targets={tuple(nutrition_targets.shape)}"
    )


imagenet_mean = torch.tensor((0.485, 0.456, 0.406)).view(1, 3, 1, 1)
imagenet_std = torch.tensor((0.229, 0.224, 0.225)).view(1, 3, 1, 1)


def show_batch(batch, title, max_images=8):
    images, class_indices, nutrition_targets = batch
    count = min(max_images, len(images))
    columns = 4
    rows = (count + columns - 1) // columns
    figure, axes = plt.subplots(rows, columns, figsize=(16, 4 * rows), squeeze=False)
    axes = axes.flatten()

    display_images = (images[:count].cpu() * imagenet_std + imagenet_mean).clamp(0, 1)
    for index in range(count):
        class_name = train_source.classes[int(class_indices[index])]
        calories, protein, carbohydrates, fat = nutrition_targets[index].tolist()
        axes[index].imshow(display_images[index].permute(1, 2, 0))
        axes[index].set_title(
            f"{class_name.replace('_', ' ').title()}\n"
            f"{calories:.0f} kcal | P {protein:.1f} g | "
            f"C {carbohydrates:.1f} g | F {fat:.1f} g",
            fontsize=9,
        )
        axes[index].axis("off")
    for axis in axes[count:]:
        axis.axis("off")
    figure.suptitle(title)
    figure.tight_layout()
    plt.show()


show_batch(train_batch, "Training batch")
show_batch(val_batch, "Validation batch")